# MAI Report Generator + PDF Report Generator

**Next layer of the Manufacturing Agentic AI (MAI) pipeline, built on top of the existing Researcher Agent.**

## What already exists (Researcher Agent) — inspected, reused, not rewritten

The provided Researcher Agent code (deterministic, no LLM required for query planning) already does:

1. Loads schema JSON for `video_analytics` (52 tables) and `mes` (108 tables).
2. Cleans/normalizes column name strings (`parse_and_clean_schema`, `clean_column_name`).
3. Maintains `DOMAIN_VOCABULARY` for keyword → table routing per database.
4. Extracts keywords from a natural-language query (`extract_tokens`, `find_matched_keywords`).
5. Selects relevant databases, tables (`score_table`) and columns (`get_relevant_columns`).
6. Detects a date/time column per table (`detect_date_column`) and simple relative-date filters
   (today / yesterday / last week / last month) via `extract_time_filter`.
7. Generates SQL (`generate_sql_for_table`, `generate_sql_queries`) — **but does not execute it.**
8. Runs a lightweight SQL sanity check (`validate_sql`, `analyze_pipeline_result`).
9. Has a 15-query global test suite (`run_global_test_suite`) that passed cleanly on all 15 cases in the
   provided test log (12 with real data selected, 2 correctly returning nothing for off-topic queries).

**This notebook does not duplicate any of the above.** Cells 6 wraps and reuses the existing functions
via `run_researcher_pipeline()` / `adapt_researcher_output()`.

## Architecture implemented in this notebook

```
Researcher Agent (existing, reused)
    -> validated research/query information      (generate_sql_queries + analyze_pipeline_result)
    -> database execution result                  (adapter accepts this once a DB layer exists;
                                                     until then, sources are marked
                                                     "query_generated_not_executed" — nothing is faked)
    -> normalized research data                    (NormalizedResearchResult contract, Cells 6-7)
    -> deterministic analytics                      (data quality + metrics engines, Cells 8-9)
    -> report generation                             (report-type detection, chart selection/data, Cells 10-12)
    -> report JSON                                    (Pydantic-validated ReportDocument, Cells 15-17)
    -> charts/tables                                   (matplotlib PNGs + structured table rows, Cell 18)
    -> PDF                                              (ReportLab, Cell 19)
```

## Design rules followed throughout this notebook

- **The Report Generator never executes arbitrary DB queries itself.** It consumes normalized research
  results; execution is the Researcher Agent / database layer's job (see `adapt_researcher_output`).
- **Every numeric KPI is computed with pandas**, never invented or computed by an LLM.
- **`USE_LLM = False` by default.** The entire pipeline — including the executive summary, key findings
  and recommendations — produces a complete, valid report with zero LLM calls. The optional LLM stage
  (Cell 14) is only for prose, is provider-agnostic via LiteLLM, and falls back to the deterministic
  summary on any failure.
- **Data quality is never silently "fixed."** Zero/negative values are flagged as anomalies needing
  verification — never declared "impossible," because no per-machine physical limits were provided in
  the schema/config files.
- **Chart data is always structured** (`[{"timestamp": ..., "value": ...}, ...]`), never a stringified
  `"12,23,34"` coordinate blob.

## Known limitations of the current Researcher Agent (see Cell 23 for full detail)

- SQL generation is not validated per-dialect (SQL Server vs. PostgreSQL quoting/paging differences).
- No table relationships/joins are available in the schema files, so cross-table/cross-database
  reporting currently relies on independently-queried, per-table result sets rather than joined data.
- InfluxDB/time-series data has no query mechanism in the Researcher Agent yet; this notebook treats
  it as a distinct source type in the normalized contract but does not generate Flux queries.
- The Researcher Agent does not execute queries against a live database yet — see `adapt_researcher_output`.

## Notebook structure

Cells 2–20 build the pipeline stage by stage (config → models → synthetic data → adapter → normalization
→ data quality → metrics → report type → charts → LLM layer → assembly → validation → rendering → PDF).
Cell 21 runs a full end-to-end demo. Cell 22 is a self-contained test suite. Cell 23 documents integration
steps and the remaining production work.


## Imports

In [1]:
import json
import os
import uuid
import warnings
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional, Tuple

import pandas as pd
import numpy as np

import matplotlib
matplotlib.use("Agg")  # headless rendering, safe for notebook -> PNG export
import matplotlib.pyplot as plt

from pydantic import BaseModel, Field, ValidationError, ConfigDict

# ReportLab (PDF generation)
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table as RLTable, TableStyle,
    Image as RLImage, PageBreak, KeepTogether, HRFlowable,
)
from reportlab.lib.enums import TA_CENTER, TA_LEFT

print("Imports OK")


Imports OK


## Configuration

In [2]:
# =====================================================================
# CONFIGURATION
# =====================================================================
from dotenv import load_dotenv
from urllib.parse import quote_plus


def _resolve_repo_root() -> Path:
    candidates = [
        Path.cwd().resolve(),
        *Path.cwd().resolve().parents,
    ]
    for candidate in candidates:
        candidate_envs = [
            candidate / ".env",
            candidate / "backend" / ".env",
            candidate / "jupyter notebooks" / ".env",
            candidate / "jupyter_notebooks" / ".env",
        ]
        for env_path in candidate_envs:
            if env_path.exists():
                if "backend" in str(env_path) or "jupyter notebooks" in str(env_path):
                    load_dotenv(env_path, override=False)
                    return candidate
                if (candidate / "backend").exists():
                    load_dotenv(candidate / "backend" / ".env", override=False)
                    return candidate
    return candidates[0]


def _load_project_env() -> List[Path]:
    loaded: List[Path] = []
    cwd = Path.cwd().resolve()
    search_roots = [cwd, *cwd.parents]
    seen: set = set()
    for root in search_roots:
        for env_path in [
            root / ".env",
            root / "backend" / ".env",
            root / "jupyter notebooks" / ".env",
            root / "jupyter_notebooks" / ".env",
        ]:
            if env_path.exists() and env_path not in seen:
                load_dotenv(env_path, override=False)
                loaded.append(env_path)
                seen.add(env_path)
    return loaded


REPO_ROOT = _resolve_repo_root()
loaded_env_files = _load_project_env()

OUTPUT_DIR = Path("./outputs/report_generator")
CHARTS_DIR = OUTPUT_DIR / "charts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

REPORT_JSON_PATH = OUTPUT_DIR / "report.json"
PDF_PATH = OUTPUT_DIR / "MAI_Report.pdf"

# ---------------------------------------------------------------------
# Real env-driven LLM setup. Prefer Gemini, then Grok, then Groq.
# This notebook now uses genuine environment variables from the MAI setup.
# ---------------------------------------------------------------------
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
GROK_API_KEY = os.environ.get("GROK_API_KEY") or os.environ.get("XAI_API_KEY")
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

if GEMINI_API_KEY:
    selected_model = os.environ.get("GEMINI_MODEL", "gemini/gemini-3.5-flash")
    selected_api_key = GEMINI_API_KEY
elif GROK_API_KEY:
    selected_model = os.environ.get("GROK_MODEL", "xai/grok-2-latest")
    selected_api_key = GROK_API_KEY
elif GROQ_API_KEY:
    selected_model = os.environ.get("GROQ_MODEL", "groq/openai/gpt-oss-20b")
    selected_api_key = GROQ_API_KEY
else:
    selected_model = os.environ.get("MAI_LLM_MODEL", "gemini/gemini-3.5-flash")
    selected_api_key = os.environ.get("MAI_LLM_API_KEY")

USE_LLM = bool(selected_api_key)

LLM_CONFIG = {
    "model": os.environ.get("MAI_LLM_MODEL") or selected_model,
    "api_base": os.environ.get("MAI_LLM_API_BASE"),
    "api_key": os.environ.get("MAI_LLM_API_KEY") or selected_api_key,
    "temperature": float(os.environ.get("LLM_TEMPERATURE", "0.2")),
    "max_tokens": int(os.environ.get("LLM_MAX_TOKENS", "800")),
    "timeout_seconds": int(os.environ.get("LLM_TIMEOUT", "30")),
}

REPORT_VERSION = "1.0"
DEFAULT_TIMEZONE = "Asia/Kolkata"
REPORT_AGENT_NAME = "report_generator"

STALE_DATA_MINUTES = 60
DUPLICATE_TS_WARNING_RATIO = 0.05
MISSING_VALUE_WARNING_RATIO = 0.10

print(f"Loaded env files: {[str(p) for p in loaded_env_files]}")
print(f"Output dir  : {OUTPUT_DIR.resolve()}")
print(f"USE_LLM     : {USE_LLM}")
print(f"LLM model   : {LLM_CONFIG['model']}")

Loaded env files: ['C:\\Users\\soroj\\OneDrive\\Desktop\\IIIOT Activate Projects\\MANUFACTURING-AGENTIC-AI\\jupyter notebooks\\.env', 'C:\\Users\\soroj\\OneDrive\\Desktop\\IIIOT Activate Projects\\MANUFACTURING-AGENTIC-AI\\backend\\.env']
Output dir  : C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\jupyter notebooks\outputs\report_generator
USE_LLM     : True
LLM model   : gemini/gemini-3.8-flash


## Data Models / Pydantic Schemas

In [3]:
# =====================================================================
# DATA MODELS / PYDANTIC SCHEMAS
#
# Two contracts:
#   1) NormalizedResearchResult - the common shape ANY source (MES/SQL
#      Server, Video Analytics/Postgres, InfluxDB) is converted into
#      before it reaches the Report Generator.
#   2) ReportDocument - the frontend-ready report JSON contract.
# =====================================================================

# --------------------------- Normalized research result ---------------

class SourceInfo(BaseModel):
    """Provenance record for one queried source/table."""
    model_config = ConfigDict(extra="allow")

    database: str
    source_name: str
    record_count: int = 0
    status: Literal["success", "query_generated_not_executed", "error", "empty"] = "empty"
    error: Optional[str] = None


class NormalizedResearchResult(BaseModel):
    """Common internal contract fed into the Report Generator."""
    model_config = ConfigDict(extra="allow")

    research_id: str
    user_query: str
    sources: List[SourceInfo] = Field(default_factory=list)
    data: List[Dict[str, Any]] = Field(default_factory=list)
    columns: List[str] = Field(default_factory=list)
    metadata: Dict[str, Any] = Field(default_factory=dict)
    data_quality: Dict[str, Any] = Field(default_factory=dict)


# --------------------------- Report JSON contract -----------------------

class TimeRange(BaseModel):
    start: Optional[str] = None
    end: Optional[str] = None
    timezone: str = DEFAULT_TIMEZONE


class ReportMeta(BaseModel):
    report_id: str
    title: str
    report_type: Literal[
        "machine_performance", "maintenance", "production",
        "safety_video_analytics", "cross_source_asset", "executive_summary",
    ]
    status: Literal["success", "partial", "error"] = "success"
    generated_at: str
    time_range: TimeRange


class KeyFinding(BaseModel):
    title: str
    description: str
    severity: Literal["info", "warning", "critical"] = "info"
    evidence_refs: List[str] = Field(default_factory=list)


class Recommendation(BaseModel):
    title: str
    description: str
    priority: Literal["low", "medium", "high"] = "medium"
    action_type: Literal["monitoring", "investigation", "maintenance", "review"] = "monitoring"


class Summary(BaseModel):
    executive_summary: str
    key_findings: List[KeyFinding] = Field(default_factory=list)
    recommendations: List[Recommendation] = Field(default_factory=list)


class Comparison(BaseModel):
    type: Literal["previous_period", "baseline", "target", "none"] = "none"
    value: Optional[float] = None
    unit: Optional[str] = None
    direction: Literal["increase", "decrease", "stable", "unknown"] = "unknown"


class KPI(BaseModel):
    id: str
    label: str
    value: Optional[float] = None
    unit: Optional[str] = None
    status: Literal["normal", "warning", "critical", "unknown"] = "unknown"
    comparison: Optional[Comparison] = None
    source_ref: Optional[str] = None  # provenance: table/source this KPI came from


class AxisSpec(BaseModel):
    field: str
    label: str
    data_type: Literal["datetime", "number", "category"] = "number"
    unit: Optional[str] = None


class SeriesSpec(BaseModel):
    id: str
    name: str
    field: str
    unit: Optional[str] = None


class Chart(BaseModel):
    id: str
    chart_type: Literal["line", "bar", "pie", "table", "kpi"]
    title: str
    description: str = ""
    x_axis: Optional[AxisSpec] = None
    y_axis: Optional[AxisSpec] = None
    series: List[SeriesSpec] = Field(default_factory=list)
    data: List[Dict[str, Any]] = Field(default_factory=list)
    insights: List[str] = Field(default_factory=list)


class ReportTable(BaseModel):
    id: str
    title: str
    columns: List[str] = Field(default_factory=list)
    rows: List[Dict[str, Any]] = Field(default_factory=list)


class DataQuality(BaseModel):
    status: Literal["complete", "partial", "poor", "unknown"] = "unknown"
    source_records: int = 0
    missing_values: int = 0
    invalid_values: int = 0
    warnings: List[str] = Field(default_factory=list)


class ReportMetadata(BaseModel):
    agent: str = REPORT_AGENT_NAME
    research_id: str
    report_version: str = REPORT_VERSION
    calculation_notes: List[str] = Field(default_factory=list)
    llm_used: bool = False


class ReportDocument(BaseModel):
    """The full, frontend-ready report JSON contract."""
    report: ReportMeta
    summary: Summary
    kpis: List[KPI] = Field(default_factory=list)
    charts: List[Chart] = Field(default_factory=list)
    tables: List[ReportTable] = Field(default_factory=list)
    data_quality: DataQuality
    sources: List[SourceInfo] = Field(default_factory=list)
    metadata: ReportMetadata


print("Pydantic models defined:",
      [m.__name__ for m in [
          SourceInfo, NormalizedResearchResult, TimeRange, ReportMeta, KeyFinding,
          Recommendation, Summary, Comparison, KPI, AxisSpec, SeriesSpec, Chart,
          ReportTable, DataQuality, ReportMetadata, ReportDocument]])


Pydantic models defined: ['SourceInfo', 'NormalizedResearchResult', 'TimeRange', 'ReportMeta', 'KeyFinding', 'Recommendation', 'Summary', 'Comparison', 'KPI', 'AxisSpec', 'SeriesSpec', 'Chart', 'ReportTable', 'DataQuality', 'ReportMetadata', 'ReportDocument']


## Sample Normalized Researcher Result — SYNTHETIC/DEMO Data

In [7]:
# =====================================================================
# LIVE DATA LOADER (REAL ENV-BACKED DATA ONLY)
# =====================================================================

import os
import warnings
from pathlib import Path
from typing import Any, Dict, List
from urllib.parse import quote_plus

from dotenv import load_dotenv
from sqlalchemy import create_engine, text


# ---------------------------------------------------------------------
# 1. Reload environment for all supported data sources
# ---------------------------------------------------------------------

def _reload_environment_for_data_sources() -> None:
    """
    Ensure the current notebook process sees the repo env files
    before DB queries run.
    """
    search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]

    for root in search_roots:
        for candidate in [
            root / ".env",
            root / "backend" / ".env",
            root / "jupyter notebooks" / ".env",
        ]:
            if candidate.exists():
                load_dotenv(candidate, override=False)


# ---------------------------------------------------------------------
# 2. Load environment-backed database configuration
# ---------------------------------------------------------------------

def _load_env_db_urls() -> Dict[str, str]:
    _reload_environment_for_data_sources()

    # ---------------- MES / SQL Server ----------------
    mes_driver = os.environ.get(
        "DB_DRIVER",
        "ODBC Driver 18 for SQL Server"
    )
    mes_server = os.environ.get(
        "DB_SERVER",
        "localhost,1433"
    )
    mes_db = os.environ.get(
        "DB_NAME",
        "mes_new"
    )

    mes_conn = (
        f"DRIVER={{{mes_driver}}};"
        f"SERVER={mes_server};"
        f"DATABASE={mes_db};"
        f"Trusted_Connection={os.environ.get('DB_TRUSTED_CONNECTION', 'yes')};"
        f"Encrypt={os.environ.get('DB_ENCRYPT', 'no')};"
        f"TrustServerCertificate={os.environ.get('DB_TRUST_SERVER_CERTIFICATE', 'yes')};"
    )

    return {
        "mes": "mssql+pyodbc:///?odbc_connect=" + quote_plus(mes_conn),

        # ---------------- PostgreSQL / Video ----------------
        "video": (
            os.environ.get("CONSTRUCTION_DB_URL", "")
            or os.environ.get("VIDEO_DB_URL", "")
        ),

        # ---------------- InfluxDB ----------------
        "influx_url": os.environ.get("INFLUXDB_URL", ""),
        "influx_token": os.environ.get("INFLUXDB_TOKEN", ""),
        "influx_org": os.environ.get("INFLUXDB_ORG", ""),
        "influx_bucket": os.environ.get("INFLUXDB_BUCKET", ""),
    }


# ---------------------------------------------------------------------
# 3. Fetch MES live data
# ---------------------------------------------------------------------

def _fetch_real_mes_rows(limit: int = 25) -> List[Dict[str, Any]]:
    urls = _load_env_db_urls()

    try:
        limit = max(1, int(limit))

        engine = create_engine(
            urls["mes"],
            pool_pre_ping=True,
            future=True
        )

        # SQL Server TOP must be part of the query structure.
        query = text(
            f"""
            SELECT TOP {limit} *
            FROM WorkOrder
            ORDER BY CreatedDate DESC
            """
        )

        with engine.connect() as conn:
            rows = conn.execute(query).mappings().all()

        return [dict(r) for r in rows]

    except Exception as exc:
        warnings.warn(f"MES live query failed: {exc}")
        return []


# ---------------------------------------------------------------------
# 4. Fetch Video Analytics live data
# ---------------------------------------------------------------------

def _fetch_real_video_rows(limit: int = 25) -> List[Dict[str, Any]]:
    urls = _load_env_db_urls()

    if not urls["video"]:
        warnings.warn(
            "Video Analytics DB URL is not configured."
        )
        return []

    try:
        limit = max(1, int(limit))

        engine = create_engine(
            urls["video"],
            pool_pre_ping=True,
            future=True
        )

        query = text(
            f"""
            SELECT *
            FROM video_events
            ORDER BY event_time DESC
            LIMIT {limit}
            """
        )

        with engine.connect() as conn:
            rows = conn.execute(query).mappings().all()

        return [dict(r) for r in rows]

    except Exception as exc:
        warnings.warn(
            f"Video live query failed: {exc}"
        )
        return []


# ---------------------------------------------------------------------
# 5. Fetch InfluxDB live data
# ---------------------------------------------------------------------

def _fetch_real_influx_rows(
    hours: int = 12,
    measurement: str = "temperature"
) -> List[Dict[str, Any]]:

    urls = _load_env_db_urls()

    required = [
        urls["influx_url"],
        urls["influx_token"],
        urls["influx_org"],
        urls["influx_bucket"],
    ]

    if not all(required):
        warnings.warn(
            "InfluxDB configuration is incomplete."
        )
        return []

    try:
        from influxdb_client import InfluxDBClient

        hours = max(1, int(hours))

        client = InfluxDBClient(
            url=urls["influx_url"],
            token=urls["influx_token"],
            org=urls["influx_org"],
        )

        # NOTE:
        # This keeps your current logic of retrieving the latest value
        # from the requested measurement.
        query = f'''
        from(bucket: "{urls["influx_bucket"]}")
          |> range(start: -{hours}h)
          |> filter(fn: (r) => r._measurement == "{measurement}")
          |> last()
        '''

        tables = client.query_api().query(query)

        rows: List[Dict[str, Any]] = []

        for table in tables:
            for record in table.records:

                row = {
                    "timestamp": (
                        record.get_time().isoformat()
                        if record.get_time()
                        else None
                    ),
                    "measurement": record.get_measurement(),
                    "field": record.get_field(),
                    "value": record.get_value(),
                }

                # Preserve useful Influx tags/metadata.
                if hasattr(record, "values") and isinstance(record.values, dict):
                    for key, value in record.values.items():
                        if key.startswith("_"):
                            continue

                        if key not in row:
                            row[key] = value

                rows.append(row)

        client.close()

        return rows

    except Exception as exc:
        warnings.warn(
            f"InfluxDB live query failed: {exc}"
        )
        return []


# ---------------------------------------------------------------------
# 6. Build normalized live research result
# ---------------------------------------------------------------------

def build_live_normalized_result(
    report_type: str,
    research_id: str = "live_research_001",
) -> Dict[str, Any]:

    _reload_environment_for_data_sources()

    urls = _load_env_db_urls()

    sources: List[Dict[str, Any]] = []
    rows: List[Dict[str, Any]] = []

    # ---------------------------------------------------------------
    # Helper to normalize rows from each source
    # ---------------------------------------------------------------

    def add_rows(
        database: str,
        source_name: str,
        values: List[Dict[str, Any]]
    ) -> None:

        values = values or []

        for item in values:
            row = dict(item)

            # Internal provenance metadata
            row["__source_database"] = database
            row["__source_table"] = source_name

            rows.append(row)

        sources.append({
            "database": database,
            "source_name": source_name,
            "record_count": len(values),
            "status": "success" if values else "empty",
        })

    # ---------------------------------------------------------------
    # InfluxDB / IoT telemetry
    # ---------------------------------------------------------------

    if report_type in {
        "machine_performance",
        "production",
        "cross_source_asset",
        "executive_summary",
    }:

        add_rows(
            "influxdb",
            "machine_telemetry",
            _fetch_real_influx_rows(
                hours=24
                if report_type == "machine_performance"
                else 12
            ),
        )

    # ---------------------------------------------------------------
    # MES
    # ---------------------------------------------------------------

    if report_type in {
        "maintenance",
        "production",
        "cross_source_asset",
        "executive_summary",
    }:

        add_rows(
            "mes",
            "WorkOrder",
            _fetch_real_mes_rows(limit=25),
        )

    # ---------------------------------------------------------------
    # Video Analytics
    # ---------------------------------------------------------------

    if report_type in {
        "safety_video_analytics",
        "cross_source_asset",
        "executive_summary",
    }:

        add_rows(
            "video_analytics",
            "hse_rule_events",
            _fetch_real_video_rows(limit=25),
        )

    # ---------------------------------------------------------------
    # No live data -> fail clearly
    # ---------------------------------------------------------------

    if not rows:

        details = {
            "DB_SERVER": os.environ.get("DB_SERVER"),
            "DB_NAME": os.environ.get("DB_NAME"),
            "CONSTRUCTION_DB_URL": bool(
                os.environ.get("CONSTRUCTION_DB_URL")
            ),
            "VIDEO_DB_URL": bool(
                os.environ.get("VIDEO_DB_URL")
            ),
            "INFLUXDB_URL": bool(
                os.environ.get("INFLUXDB_URL")
            ),
            "GEMINI_API_KEY": bool(
                os.environ.get("GEMINI_API_KEY")
            ),
            "GROK_API_KEY": bool(
                os.environ.get("GROK_API_KEY")
                or os.environ.get("XAI_API_KEY")
            ),
            "GROQ_API_KEY": bool(
                os.environ.get("GROQ_API_KEY")
            ),
        }

        raise RuntimeError(
            "No real rows were available from the configured "
            "environment-backed databases. "
            "Verify DATABASE_URL, CONSTRUCTION_DB_URL, "
            "VIDEO_DB_URL, INFLUXDB_* and required credentials "
            "before generating the PDF. "
            f"Connection details loaded: {details}"
        )

    # ---------------------------------------------------------------
    # User query mapping
    # ---------------------------------------------------------------

    user_queries = {
        "machine_performance": (
            "Generate a machine performance report "
            "for the latest live plant data"
        ),

        "maintenance": (
            "Show live maintenance / work-order status "
            "from the MES system"
        ),

        "production": (
            "Production snapshot from live MES + telemetry data"
        ),

        "safety_video_analytics": (
            "Safety and video analytics overview "
            "from live camera events"
        ),

        "cross_source_asset": (
            "Cross-source asset report using live telemetry, "
            "MES, and video data"
        ),

        "executive_summary": (
            "Executive summary using live plant telemetry "
            "and operational data"
        ),
    }

    # ---------------------------------------------------------------
    # Final normalized result
    # ---------------------------------------------------------------

    normalized_result = {
        "research_id": research_id,

        "user_query": user_queries.get(
            report_type,
            "Live operational report"
        ),

        "sources": sources,

        "data": rows,

        "columns": sorted({
            key
            for row in rows
            for key in row.keys()
            if not key.startswith("__")
        }),

        "metadata": {
            "demo": False,
            "note": "REAL DATA from configured MAI DB sources",
            "report_type": report_type,
        },

        "data_quality": {},
    }

    return normalized_result


# ---------------------------------------------------------------------
# 7. Smoke Check
# ---------------------------------------------------------------------

_live = None

try:

    _live = build_live_normalized_result(
        "machine_performance"
    )

    print(
        f"Live sources: "
        f"{[s['source_name'] for s in _live['sources']]}, "
        f"rows={len(_live['data'])}"
    )

except Exception as exc:

    print(
        f"[INFO] No live report data available: {exc}"
    )

[INFO] No live report data available: No real rows were available from the configured environment-backed databases. Verify DATABASE_URL, CONSTRUCTION_DB_URL, VIDEO_DB_URL, INFLUXDB_* and required credentials before generating the PDF. Connection details loaded: {'DB_SERVER': 'localhost,1433', 'DB_NAME': 'mes_new', 'CONSTRUCTION_DB_URL': True, 'VIDEO_DB_URL': False, 'INFLUXDB_URL': True, 'GEMINI_API_KEY': True, 'GROK_API_KEY': False, 'GROQ_API_KEY': True}


## Adapter for the REAL Researcher Agent Output

In [8]:
# =====================================================================
# ADAPTER FOR THE REAL RESEARCHER AGENT OUTPUT
#
# Reuses the EXISTING Researcher Agent as-is (no rewrite). Today that
# agent only plans/generates SQL -- it does not execute queries against
# SQL Server / PostgreSQL. So this adapter keeps two things explicitly
# separate, per the design requirement:
#
#   1) generated/validated queries  -> researcher_output (from
#      generate_sql_queries()/analyze_pipeline_result())
#   2) actual database execution results -> execution_results, supplied
#      separately once a DB-execution layer exists. Until then, sources
#      are marked status="query_generated_not_executed" and never faked.
#
# Set RESEARCHER_MODULE_NAME below to match your actual Researcher Agent
# file (e.g. "researcher_agent") once it is importable from this
# notebook's working directory / PYTHONPATH.
# =====================================================================

RESEARCHER_MODULE_NAME = "researcher_agent"  # <-- adjust to your real module filename

try:
    _researcher = __import__(RESEARCHER_MODULE_NAME)
    RESEARCHER_AVAILABLE = True
    print(f"Loaded real Researcher Agent module: {RESEARCHER_MODULE_NAME}")
except ImportError:
    _researcher = None
    RESEARCHER_AVAILABLE = False
    print(f"[INFO] '{RESEARCHER_MODULE_NAME}' not importable here -- "
          f"adapter is defined and ready, but this demo will fall back to synthetic data.")


def run_researcher_pipeline(user_query: str) -> Dict[str, Any]:
    """
    Thin wrapper around the EXISTING Researcher Agent functions:
    customize_and_split_query -> generate_sql_queries -> analyze_pipeline_result.
    Returns the generate_sql_queries() output (richest structure: table + SQL + columns).
    Raises RuntimeError if the real module isn't available.
    """
    if not RESEARCHER_AVAILABLE:
        raise RuntimeError(
            f"Researcher module '{RESEARCHER_MODULE_NAME}' is not importable. "
            "Place it on the path or update RESEARCHER_MODULE_NAME."
        )
    split = _researcher.customize_and_split_query(user_query)
    sql_result = _researcher.generate_sql_queries(split)
    validation_report = _researcher.analyze_pipeline_result(sql_result)
    return {"split": split, "sql_result": sql_result, "validation_report": validation_report}


def adapt_researcher_output(
    researcher_sql_result: Dict[str, Any],
    execution_results: Optional[Dict[str, List[Dict[str, Any]]]] = None,
    research_id: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Map the EXISTING Researcher Agent's `generate_sql_queries()` output
    into the NormalizedResearchResult contract.

    Parameters
    ----------
    researcher_sql_result : the dict returned by generate_sql_queries()
        (raw_user_query, time_filter_applied, databases[*].queries[*]).
    execution_results : optional dict keyed by "{database}.{table}" (or
        just "{table}") -> list[dict] of ACTUAL rows returned by running
        the SQL against the real database. When omitted (the current
        state of the project), every source is marked
        "query_generated_not_executed" and `data` stays empty -- nothing
        is invented.
    research_id : optional id; generated if not provided.
    """
    research_id = research_id or f"research_{uuid.uuid4().hex[:8]}"
    execution_results = execution_results or {}

    sources: List[Dict[str, Any]] = []
    data: List[Dict[str, Any]] = []
    sql_by_table: Dict[str, str] = {}
    all_columns: set = set()

    for db_entry in researcher_sql_result.get("databases", []):
        database = db_entry.get("database", "unknown")
        for q in db_entry.get("queries", []):
            table = q.get("table", "unknown_table")
            sql_by_table[f"{database}.{table}"] = q.get("sql", "")

            rows = execution_results.get(f"{database}.{table}") or execution_results.get(table)
            if rows is not None:
                status = "success" if rows else "empty"
                for r in rows:
                    r = dict(r)
                    r["__source_database"] = database
                    r["__source_table"] = table
                    data.append(r)
                    all_columns.update(k for k in r if not k.startswith("__"))
            else:
                status = "query_generated_not_executed"
                rows = []

            sources.append({
                "database": database,
                "source_name": table,
                "record_count": len(rows),
                "status": status,
            })

    return {
        "research_id": research_id,
        "user_query": researcher_sql_result.get("raw_user_query", ""),
        "sources": sources,
        "data": data,
        "columns": sorted(all_columns),
        "metadata": {
            "time_filter_applied": researcher_sql_result.get("time_filter_applied"),
            "sql_by_table": sql_by_table,
            "execution_layer_available": bool(execution_results),
        },
        "data_quality": {},
    }


print("Adapter functions ready: run_researcher_pipeline(), adapt_researcher_output()")


[INFO] 'researcher_agent' not importable here -- adapter is defined and ready, but this demo will fall back to synthetic data.
Adapter functions ready: run_researcher_pipeline(), adapt_researcher_output()


## Data Normalization

In [9]:
# =====================================================================
# DATA NORMALIZATION
# Turns NormalizedResearchResult["data"] (a flat list of dicts tagged
# with __source_database / __source_table) into one pandas DataFrame
# per source, with a best-effort timestamp column detected + parsed.
# =====================================================================

_TIMESTAMP_NAME_HINTS = (
    "timestamp", "created_at", "updated_at", "checked_at", "triggered_at",
    "created_date", "updated_date", "startdate", "enddate", "date",
    "changedat", "lastupdated", "time_in", "time_out",
)


def detect_timestamp_column(df: pd.DataFrame) -> Optional[str]:
    """Best-effort timestamp column detection for an arbitrary source DataFrame."""
    cols_lower = {c.lower(): c for c in df.columns}
    for hint in _TIMESTAMP_NAME_HINTS:
        if hint in cols_lower:
            return cols_lower[hint]
    for c in df.columns:
        cl = c.lower()
        if any(h in cl for h in ("_at", "_on", "date", "time")) and len(cl) > 3:
            return c
    return None


def normalize_research_result(normalized_result: Dict[str, Any]) -> Dict[str, Any]:
    """
    Split the flat `data` list by (__source_database, __source_table) and
    build one DataFrame per source. Parses/coerces the detected timestamp
    column to pandas datetime (errors='coerce' -> NaT, never silently
    dropped -- NaT counts feed the data-quality engine).
    """
    rows = normalized_result.get("data", [])
    df_all = pd.DataFrame(rows)

    source_frames: Dict[str, pd.DataFrame] = {}
    timestamp_cols: Dict[str, Optional[str]] = {}

    if df_all.empty:
        for s in normalized_result.get("sources", []):
            key = f"{s['database']}.{s['source_name']}"
            source_frames[key] = pd.DataFrame()
            timestamp_cols[key] = None
        return {
            "source_frames": source_frames,
            "timestamp_cols": timestamp_cols,
            "research_id": normalized_result.get("research_id"),
            "user_query": normalized_result.get("user_query"),
        }

    group_cols = ["__source_database", "__source_table"]
    for (database, table), sub in df_all.groupby(group_cols, dropna=False):
        key = f"{database}.{table}"
        sub = sub.drop(columns=[c for c in group_cols if c in sub.columns]).reset_index(drop=True)
        ts_col = detect_timestamp_column(sub)
        if ts_col:
            sub[ts_col] = pd.to_datetime(sub[ts_col], errors="coerce", utc=True)
            sub = sub.sort_values(ts_col)
        source_frames[key] = sub
        timestamp_cols[key] = ts_col

    return {
        "source_frames": source_frames,
        "timestamp_cols": timestamp_cols,
        "research_id": normalized_result.get("research_id"),
        "user_query": normalized_result.get("user_query"),
    }


# Smoke-check
_norm = normalize_research_result(_demo)
for k, df in _norm["source_frames"].items():
    print(k, df.shape, "timestamp_col=", _norm["timestamp_cols"][k])


NameError: name '_demo' is not defined

## Data Quality Engine

In [ ]:
# =====================================================================
# DATA QUALITY ENGINE
#
# Flags concerns WITHOUT silently "fixing" or reinterpreting industrial
# data. Zero/negative values are flagged as anomalies needing
# verification, NOT declared impossible -- we have no per-machine
# physical limits in the provided schema/config.
# =====================================================================

def _numeric_columns(df: pd.DataFrame) -> List[str]:
    return [c for c in df.select_dtypes(include=[np.number]).columns]


def assess_source_quality(source_key: str, df: pd.DataFrame, ts_col: Optional[str]) -> Dict[str, Any]:
    """Per-source quality assessment. Returns warnings + counts, never raises."""
    warnings_list: List[str] = []
    missing_values = 0
    invalid_values = 0

    if df.empty:
        return {
            "source": source_key, "record_count": 0,
            "warnings": [f"'{source_key}' returned 0 records."],
            "missing_values": 0, "invalid_values": 0,
        }

    # Missing values across all columns
    na_counts = df.isna().sum()
    total_cells = df.shape[0] * df.shape[1]
    missing_values = int(na_counts.sum())
    if total_cells and (missing_values / total_cells) > MISSING_VALUE_WARNING_RATIO:
        warnings_list.append(
            f"'{source_key}': {missing_values} missing cell(s) "
            f"({missing_values/total_cells:.1%} of all cells)."
        )

    # Timestamp-specific checks
    if ts_col and ts_col in df.columns:
        na_ts = int(df[ts_col].isna().sum())
        if na_ts:
            warnings_list.append(f"'{source_key}': {na_ts} row(s) with missing/unparseable timestamp in '{ts_col}'.")
            invalid_values += na_ts

        # If an entity/id column exists (e.g. machine_id), duplicate timestamps are
        # expected across different entities -- check duplicates WITHIN each entity.
        id_col = next((c for c in df.columns if c.lower().endswith("_id") and c != ts_col), None)
        dup_subset = [id_col, ts_col] if id_col else [ts_col]
        dup_ts = int(df.duplicated(subset=dup_subset).sum())
        if len(df) and (dup_ts / len(df)) > DUPLICATE_TS_WARNING_RATIO:
            scope = f" per '{id_col}'" if id_col else ""
            warnings_list.append(f"'{source_key}': {dup_ts} duplicate timestamp(s) in '{ts_col}'{scope}.")

        valid_ts = df[ts_col].dropna()
        if not valid_ts.empty:
            latest = valid_ts.max()
            now = pd.Timestamp.now(tz="UTC")
            age_minutes = (now - latest).total_seconds() / 60.0
            if age_minutes > STALE_DATA_MINUTES:
                warnings_list.append(
                    f"'{source_key}': latest record is {age_minutes:.0f} min old "
                    f"(stale threshold = {STALE_DATA_MINUTES} min)."
                )

    # Duplicate full rows
    dup_rows = int(df.duplicated().sum())
    if dup_rows:
        warnings_list.append(f"'{source_key}': {dup_rows} fully duplicate row(s).")

    # Zero / negative numeric values -> flagged as anomaly, not invalid
    for col in _numeric_columns(df):
        zero_n = int((df[col] == 0).sum())
        neg_n = int((df[col] < 0).sum())
        if zero_n and zero_n / len(df) > 0.2:
            warnings_list.append(
                f"'{source_key}.{col}': {zero_n} zero value(s) -- possible offline sensor, verify against source."
            )
        if neg_n:
            warnings_list.append(
                f"'{source_key}.{col}': {neg_n} negative value(s) -- flagged as anomaly "
                f"(no domain rule available to classify as invalid)."
            )

    return {
        "source": source_key, "record_count": len(df),
        "warnings": warnings_list,
        "missing_values": missing_values, "invalid_values": invalid_values,
    }


def compute_data_quality(normalized: Dict[str, Any]) -> Dict[str, Any]:
    """Aggregate per-source quality assessments into the DataQuality contract shape."""
    all_warnings: List[str] = []
    total_records = 0
    total_missing = 0
    total_invalid = 0

    source_frames = normalized["source_frames"]
    timestamp_cols = normalized["timestamp_cols"]

    if not source_frames:
        return {
            "status": "unknown", "source_records": 0, "missing_values": 0,
            "invalid_values": 0, "warnings": ["No sources returned by the Researcher Agent."],
        }

    for key, df in source_frames.items():
        res = assess_source_quality(key, df, timestamp_cols.get(key))
        all_warnings.extend(res["warnings"])
        total_records += res["record_count"]
        total_missing += res["missing_values"]
        total_invalid += res["invalid_values"]

    if total_records == 0:
        status = "poor"
    elif len(all_warnings) == 0:
        status = "complete"
    elif len(all_warnings) <= 2:
        status = "partial"
    else:
        status = "partial" if total_records > 0 else "poor"

    return {
        "status": status,
        "source_records": total_records,
        "missing_values": total_missing,
        "invalid_values": total_invalid,
        "warnings": all_warnings,
    }


# Smoke-check
_dq = compute_data_quality(_norm)
print(json.dumps(_dq, indent=2)[:600])


## Deterministic Metrics Engine

In [ ]:
# =====================================================================
# DETERMINISTIC METRICS ENGINE
# All numbers in the final report come from here (pandas), never from
# an LLM. Every KPI keeps a `source_ref` back to its origin table.
# =====================================================================

def numeric_summary(series: pd.Series) -> Dict[str, Optional[float]]:
    s = series.dropna()
    if s.empty:
        return {"count": 0, "sum": None, "mean": None, "median": None,
                "min": None, "max": None, "std": None, "latest": None, "first": None}
    return {
        "count": int(s.count()),
        "sum": round(float(s.sum()), 3),
        "mean": round(float(s.mean()), 3),
        "median": round(float(s.median()), 3),
        "min": round(float(s.min()), 3),
        "max": round(float(s.max()), 3),
        "std": round(float(s.std()), 3) if s.count() > 1 else 0.0,
        "latest": round(float(s.iloc[-1]), 3),
        "first": round(float(s.iloc[0]), 3),
    }


def compute_trend(df: pd.DataFrame, ts_col: str, value_col: str) -> Dict[str, Any]:
    """Trend direction + percent change using first vs. last valid reading."""
    sub = df[[ts_col, value_col]].dropna().sort_values(ts_col)
    if len(sub) < 2:
        return {"direction": "unknown", "pct_change": None}
    first_val = float(sub[value_col].iloc[0])
    last_val = float(sub[value_col].iloc[-1])
    if first_val == 0:
        pct_change = None
    else:
        pct_change = round(((last_val - first_val) / abs(first_val)) * 100, 2)
    if pct_change is None:
        direction = "stable" if last_val == first_val else ("increase" if last_val > first_val else "decrease")
    elif abs(pct_change) < 1.0:
        direction = "stable"
    else:
        direction = "increase" if pct_change > 0 else "decrease"
    return {"direction": direction, "pct_change": pct_change}


def compute_threshold_violations(df: pd.DataFrame, col: str,
                                  min_val: Optional[float] = None,
                                  max_val: Optional[float] = None) -> Dict[str, Any]:
    if col not in df.columns:
        return {"count": 0, "min_val": min_val, "max_val": max_val}
    s = df[col].dropna()
    mask = pd.Series([False] * len(s), index=s.index)
    if min_val is not None:
        mask |= s < min_val
    if max_val is not None:
        mask |= s > max_val
    return {"count": int(mask.sum()), "min_val": min_val, "max_val": max_val}


def compute_status_distribution(df: pd.DataFrame, col: str) -> Dict[str, Any]:
    if col not in df.columns or df.empty:
        return {}
    counts = df[col].fillna("unknown").astype(str).value_counts()
    total = int(counts.sum())
    return {
        "counts": {k: int(v) for k, v in counts.items()},
        "percentages": {k: round(v / total * 100, 1) for k, v in counts.items()} if total else {},
    }


def compute_event_counts(df: pd.DataFrame, groupby_col: str) -> Dict[str, int]:
    if groupby_col not in df.columns or df.empty:
        return {}
    return {str(k): int(v) for k, v in df.groupby(groupby_col).size().items()}


def build_kpis_for_source(
    source_key: str, df: pd.DataFrame, ts_col: Optional[str],
) -> List[Dict[str, Any]]:
    """Auto-generate KPI dicts (matching the KPI contract) for every numeric column."""
    kpis: List[Dict[str, Any]] = []
    if df.empty:
        return kpis

    for col in _numeric_columns(df):
        stats = numeric_summary(df[col])
        if stats["count"] == 0:
            continue
        comparison = None
        status = "normal"
        if ts_col and ts_col in df.columns:
            trend = compute_trend(df, ts_col, col)
            direction = trend["direction"]
            comparison = {
                "type": "previous_period" if trend["pct_change"] is not None else "none",
                "value": trend["pct_change"],
                "unit": "%",
                "direction": direction,
            }
            if direction == "increase" and trend["pct_change"] and trend["pct_change"] > 15:
                status = "warning"
        kpis.append({
            "id": f"kpi_{source_key.replace('.', '_')}_{col}_avg",
            "label": f"Average {col.replace('_', ' ').title()} ({source_key.split('.')[-1]})",
            "value": stats["mean"],
            "unit": None,
            "status": status,
            "comparison": comparison,
            "source_ref": source_key,
        })
    return kpis


# Smoke-check
_kpis = []
for _k, _df in _norm["source_frames"].items():
    _kpis.extend(build_kpis_for_source(_k, _df, _norm["timestamp_cols"].get(_k)))
print(f"Generated {len(_kpis)} KPI(s). Example:", _kpis[0] if _kpis else None)


## Report-Type Detection

In [ ]:
# =====================================================================
# REPORT TYPE DETECTION (deterministic, keyword + source based)
# Extensible: add new (report_type, keywords, source_hints) tuples below.
# =====================================================================

REPORT_TYPE_RULES: List[Tuple[str, List[str], List[str]]] = [
    ("safety_video_analytics", ["safety", "hse", "violation", "ppe", "incident", "alert", "camera", "zone"],
     ["video_analytics"]),
    ("maintenance", ["maintenance", "work order", "workorder", "repair", "downtime", "service"],
     ["mes"]),
    ("production", ["production", "output", "throughput", "planning", "finished goods", "work order"],
     ["mes"]),
    ("machine_performance", ["machine", "utilization", "performance", "telemetry", "temperature",
                              "vibration", "rpm", "sensor"], ["influxdb", "mes"]),
]


def detect_report_type(normalized_result: Dict[str, Any]) -> str:
    """
    Deterministic report-type detection. Falls back to
    'cross_source_asset' when >1 distinct database is involved with no
    single dominant keyword match, and 'executive_summary' otherwise.
    """
    query = (normalized_result.get("user_query") or "").lower()
    databases = {s["database"] for s in normalized_result.get("sources", [])}

    scores: Dict[str, int] = {}
    for report_type, keywords, source_hints in REPORT_TYPE_RULES:
        score = sum(1 for kw in keywords if kw in query)
        score += sum(1 for sh in source_hints if sh in databases)
        if score:
            scores[report_type] = score

    # 3+ distinct databases queried together is a cross-source asset report
    # by definition, regardless of keyword overlap.
    if len(databases) >= 3:
        return "cross_source_asset"

    if scores:
        ranked = sorted(scores.items(), key=lambda kv: -kv[1])
        best_type, best_score = ranked[0]
        # Two distinct databases + no single dominant signal -> cross-source
        if len(databases) == 2 and (len(ranked) == 1 or ranked[0][1] - ranked[1][1] <= 1):
            return "cross_source_asset"
        return best_type

    if len(databases) >= 2:
        return "cross_source_asset"
    return "executive_summary"


# Smoke-check
for _rt in ["machine_performance", "maintenance", "safety_video_analytics", "cross_source_asset"]:
    _demo_rt = build_synthetic_normalized_result(_rt)
    print(_rt, "->", detect_report_type(_demo_rt))


## Chart Selection

In [ ]:
# =====================================================================
# CHART SELECTION (deterministic)
#
# Rules:
#  - datetime + numeric time-series           -> line
#  - category + aggregated numeric            -> bar
#  - category + percentage/count distribution -> pie   (<=6 categories)
#  - complex/raw tabular data                  -> table
#  - single important aggregate                -> kpi   (handled in cell 9)
#
# The LLM (if enabled) may SUGGEST a chart type via the optional insight
# layer, but this function is the source of truth: an LLM suggestion
# that doesn't fit the data's actual shape is never applied.
# =====================================================================

MAX_PIE_CATEGORIES = 6
MAX_LINE_SERIES = 4


def suggest_charts(source_key: str, df: pd.DataFrame, ts_col: Optional[str]) -> List[Dict[str, Any]]:
    """Return a list of chart_spec dicts: {id, chart_type, title, x_field, y_fields, cat_field}."""
    specs: List[Dict[str, Any]] = []
    if df.empty:
        return specs

    numeric_cols = _numeric_columns(df)
    categorical_cols = [c for c in df.columns
                         if c not in numeric_cols and c != ts_col and df[c].nunique() <= 20]

    table_short = source_key.split(".")[-1]

    # Rule 1: datetime + numeric -> line chart(s), grouped in batches of MAX_LINE_SERIES
    if ts_col and numeric_cols:
        for i in range(0, len(numeric_cols), MAX_LINE_SERIES):
            batch = numeric_cols[i:i + MAX_LINE_SERIES]
            specs.append({
                "id": f"chart_{table_short}_trend_{i}",
                "chart_type": "line",
                "title": f"{table_short.replace('_', ' ').title()} Over Time",
                "x_field": ts_col,
                "y_fields": batch,
                "cat_field": None,
            })

    # Rule 2/3: categorical + numeric -> bar (or pie if low-cardinality distribution)
    for cat_col in categorical_cols:
        n_unique = df[cat_col].nunique()
        if numeric_cols:
            specs.append({
                "id": f"chart_{table_short}_{cat_col}_by_{numeric_cols[0]}",
                "chart_type": "bar",
                "title": f"{numeric_cols[0].replace('_', ' ').title()} by {cat_col.replace('_', ' ').title()}",
                "x_field": cat_col, "y_fields": [numeric_cols[0]], "cat_field": cat_col,
            })
        if n_unique <= MAX_PIE_CATEGORIES:
            specs.append({
                "id": f"chart_{table_short}_{cat_col}_distribution",
                "chart_type": "pie",
                "title": f"{cat_col.replace('_', ' ').title()} Distribution",
                "x_field": None, "y_fields": [], "cat_field": cat_col,
            })

    # Rule 4: fallback / always provide a raw table for traceability
    specs.append({
        "id": f"chart_{table_short}_raw_table",
        "chart_type": "table",
        "title": f"{table_short.replace('_', ' ').title()} - Raw Records",
        "x_field": None, "y_fields": [], "cat_field": None,
    })

    return specs


def validate_chart_spec(spec: Dict[str, Any], df: pd.DataFrame) -> bool:
    """Backend validation: reject a chart type that doesn't fit the data (e.g. LLM-suggested)."""
    if spec["chart_type"] == "line":
        return bool(spec.get("x_field")) and bool(spec.get("y_fields"))
    if spec["chart_type"] == "bar":
        return bool(spec.get("x_field")) and bool(spec.get("y_fields"))
    if spec["chart_type"] == "pie":
        cat = spec.get("cat_field")
        return bool(cat) and cat in df.columns and df[cat].nunique() <= MAX_PIE_CATEGORIES
    if spec["chart_type"] == "table":
        return not df.empty
    return True


# Smoke-check
for _k, _df in _norm["source_frames"].items():
    _specs = [s for s in suggest_charts(_k, _df, _norm["timestamp_cols"].get(_k)) if validate_chart_spec(s, _df)]
    print(_k, "->", [(s["chart_type"], s["id"]) for s in _specs])


## Chart Data Preparation

In [ ]:
# =====================================================================
# CHART DATA PREPARATION
# Converts a chart_spec + DataFrame into the structured Chart contract
# (list of dicts, e.g. {"timestamp": ..., "temperature": ...}) -- NEVER
# stringified "x,y,z" coordinate blobs.
# =====================================================================

MAX_CHART_POINTS = 500  # keep PDFs/JSON reasonably sized; downsample if needed


def _downsample(df: pd.DataFrame, max_points: int = MAX_CHART_POINTS) -> pd.DataFrame:
    if len(df) <= max_points:
        return df
    step = max(1, len(df) // max_points)
    return df.iloc[::step]


def build_chart_data(spec: Dict[str, Any], df: pd.DataFrame) -> Dict[str, Any]:
    """Build one Chart-contract dict from a validated chart_spec."""
    chart_type = spec["chart_type"]

    if chart_type == "line":
        sub = _downsample(df[[spec["x_field"], *spec["y_fields"]]].dropna(subset=[spec["x_field"]]))
        data = []
        for _, row in sub.iterrows():
            point = {"timestamp": row[spec["x_field"]].isoformat()
                      if isinstance(row[spec["x_field"]], pd.Timestamp) else row[spec["x_field"]]}
            for y in spec["y_fields"]:
                val = row[y]
                point[y] = None if pd.isna(val) else float(val)
            data.append(point)
        return {
            "id": spec["id"], "chart_type": "line", "title": spec["title"], "description": "",
            "x_axis": {"field": spec["x_field"], "label": "Time", "data_type": "datetime"},
            "y_axis": {"field": spec["y_fields"][0], "label": spec["y_fields"][0].replace("_", " ").title(),
                       "data_type": "number"},
            "series": [{"id": y, "name": y.replace("_", " ").title(), "field": y} for y in spec["y_fields"]],
            "data": data, "insights": [],
        }

    if chart_type == "bar":
        y = spec["y_fields"][0]
        agg = df.groupby(spec["x_field"])[y].mean().reset_index()
        data = [{"category": str(r[spec["x_field"]]), "value": round(float(r[y]), 3)} for _, r in agg.iterrows()]
        return {
            "id": spec["id"], "chart_type": "bar", "title": spec["title"], "description": "",
            "x_axis": {"field": "category", "label": spec["x_field"].replace("_", " ").title(), "data_type": "category"},
            "y_axis": {"field": "value", "label": y.replace("_", " ").title(), "data_type": "number"},
            "series": [{"id": y, "name": y.replace("_", " ").title(), "field": "value"}],
            "data": data, "insights": [],
        }

    if chart_type == "pie":
        cat = spec["cat_field"]
        counts = df[cat].fillna("unknown").astype(str).value_counts()
        data = [{"category": k, "value": int(v)} for k, v in counts.items()]
        return {
            "id": spec["id"], "chart_type": "pie", "title": spec["title"], "description": "",
            "x_axis": None, "y_axis": None,
            "series": [{"id": cat, "name": cat.replace("_", " ").title(), "field": "value"}],
            "data": data, "insights": [],
        }

    if chart_type == "table":
        sub = _downsample(df, max_points=50)
        data = json.loads(sub.astype(object).where(pd.notna(sub), None).to_json(orient="records", date_format="iso"))
        return {
            "id": spec["id"], "chart_type": "table", "title": spec["title"], "description": "",
            "x_axis": None, "y_axis": None, "series": [],
            "data": data, "insights": [],
        }

    raise ValueError(f"Unsupported chart_type: {chart_type}")


def build_all_charts(normalized: Dict[str, Any]) -> List[Dict[str, Any]]:
    charts = []
    for key, df in normalized["source_frames"].items():
        ts_col = normalized["timestamp_cols"].get(key)
        for spec in suggest_charts(key, df, ts_col):
            if validate_chart_spec(spec, df):
                try:
                    charts.append(build_chart_data(spec, df))
                except Exception as e:
                    warnings.warn(f"Chart build failed for {spec['id']}: {e}")
    return charts


# Smoke-check
_charts = build_all_charts(_norm)
print(f"Built {len(_charts)} chart(s):", [(c["chart_type"], c["id"]) for c in _charts])


## Optional LiteLLM Configuration

In [ ]:
# =====================================================================
# OPTIONAL LITELLM CONFIGURATION
# Provider-agnostic on purpose (OpenAI, Anthropic, Azure, local, etc.)
# so MAI can switch providers/models without touching this notebook.
# This cell NEVER calls out to a network; it only prepares config.
# =====================================================================

try:
    import litellm
    LITELLM_AVAILABLE = True
except ImportError:
    litellm = None
    LITELLM_AVAILABLE = False


def get_llm_call_kwargs(messages: List[Dict[str, str]]) -> Dict[str, Any]:
    """Build the kwargs for litellm.completion(**kwargs) from LLM_CONFIG."""
    kwargs = {
        "model": LLM_CONFIG["model"],
        "messages": messages,
        "temperature": LLM_CONFIG["temperature"],
        "max_tokens": LLM_CONFIG["max_tokens"],
        "timeout": LLM_CONFIG["timeout_seconds"],
    }
    if LLM_CONFIG.get("api_base"):
        kwargs["api_base"] = LLM_CONFIG["api_base"]
    if LLM_CONFIG.get("api_key"):
        kwargs["api_key"] = LLM_CONFIG["api_key"]
    return kwargs


print(f"LiteLLM available: {LITELLM_AVAILABLE} | USE_LLM: {USE_LLM} | model: {LLM_CONFIG['model']}")
if USE_LLM and not LITELLM_AVAILABLE:
    warnings.warn("USE_LLM=True but litellm is not installed -- pipeline will fall back to deterministic summaries.")


## Optional LLM Insight Generation

In [ ]:
# =====================================================================
# OPTIONAL LLM INSIGHT GENERATION
#
# The LLM is used ONLY for prose (executive summary / narrative
# findings / recommendations). It never computes numbers, chart
# coordinates, tables, or record counts -- those are already final by
# this point in the pipeline (cells 8-12). Any malformed/failed LLM
# response falls back to the deterministic summary so the pipeline
# always produces a complete, valid report.
# =====================================================================

def deterministic_summary(context: Dict[str, Any]) -> Dict[str, Any]:
    """Template-based executive summary / findings / recommendations. No LLM."""
    dq = context["data_quality"]
    kpis = context["kpis"]
    report_type = context["report_type"]

    n_sources = len(context["sources"])
    total_records = dq["source_records"]

    exec_summary = (
        f"This {report_type.replace('_', ' ')} report covers {n_sources} data source(s) "
        f"and {total_records} record(s). Data quality is '{dq['status']}' "
        f"with {len(dq['warnings'])} warning(s) flagged. "
        f"{len(kpis)} KPI(s) were computed deterministically from the underlying data."
    )

    key_findings = []
    for w in dq["warnings"][:5]:
        key_findings.append({
            "title": "Data quality flag", "description": w,
            "severity": "warning", "evidence_refs": [],
        })
    for kpi in kpis:
        comp = kpi.get("comparison")
        if comp and comp.get("direction") == "increase" and comp.get("value") and comp["value"] > 15:
            key_findings.append({
                "title": f"{kpi['label']} trending up",
                "description": f"{kpi['label']} increased {comp['value']}% over the observed window.",
                "severity": "warning", "evidence_refs": [kpi["source_ref"]] if kpi.get("source_ref") else [],
            })
    if not key_findings:
        key_findings.append({
            "title": "No significant findings",
            "description": "No data-quality warnings or notable KPI trends were detected in this window.",
            "severity": "info", "evidence_refs": [],
        })

    recommendations = []
    if dq["status"] in ("partial", "poor"):
        recommendations.append({
            "title": "Review data quality warnings",
            "description": "Investigate the flagged data-quality issues before relying on this report for decisions.",
            "priority": "high" if dq["status"] == "poor" else "medium",
            "action_type": "investigation",
        })
    if total_records == 0:
        recommendations.append({
            "title": "Confirm data source connectivity",
            "description": "No records were returned. Confirm the Researcher Agent's query executed against a live source.",
            "priority": "high", "action_type": "investigation",
        })
    if not recommendations:
        recommendations.append({
            "title": "Continue routine monitoring",
            "description": "No immediate action required based on current data.",
            "priority": "low", "action_type": "monitoring",
        })

    return {
        "executive_summary": exec_summary,
        "key_findings": key_findings,
        "recommendations": recommendations,
    }


def _validate_llm_shape(obj: Dict[str, Any]) -> bool:
    return (
        isinstance(obj, dict)
        and isinstance(obj.get("executive_summary"), str)
        and isinstance(obj.get("key_findings"), list)
        and isinstance(obj.get("recommendations"), list)
    )


def generate_insights(context: Dict[str, Any]) -> Tuple[Dict[str, Any], bool]:
    """
    Returns (insights_dict, llm_used). Always deterministic when USE_LLM
    is False. When USE_LLM is True, attempts a structured LLM call and
    falls back to the deterministic summary on ANY failure (network,
    malformed JSON, missing keys, timeout, etc.) -- never crashes.
    """
    if not USE_LLM:
        return deterministic_summary(context), False

    if not LITELLM_AVAILABLE:
        warnings.warn("USE_LLM=True but litellm not installed; using deterministic summary.")
        return deterministic_summary(context), False

    prompt = (
        "You are a manufacturing analytics assistant. Given this JSON context of "
        "already-computed KPIs and data-quality results (do NOT invent new numbers), "
        "return ONLY a JSON object with keys 'executive_summary' (string), "
        "'key_findings' (list of {title, description, severity, evidence_refs}), "
        "'recommendations' (list of {title, description, priority, action_type}).\n\n"
        f"CONTEXT:\n{json.dumps(context, default=str)[:6000]}"
    )
    try:
        resp = litellm.completion(**get_llm_call_kwargs([{"role": "user", "content": prompt}]))
        raw_text = resp["choices"][0]["message"]["content"]
        raw_text = raw_text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        parsed = json.loads(raw_text)
        if not _validate_llm_shape(parsed):
            raise ValueError("LLM response missing required keys")
        return parsed, True
    except Exception as e:
        warnings.warn(f"LLM insight generation failed ({e}); falling back to deterministic summary.")
        return deterministic_summary(context), False


# Smoke-check (deterministic path)
_ctx = {
    "report_type": "machine_performance",
    "sources": _norm["source_frames"].keys(),
    "data_quality": _dq,
    "kpis": _kpis,
}
_insights, _llm_used = generate_insights(_ctx)
print("llm_used:", _llm_used)
print(_insights["executive_summary"])


## Report JSON Assembly

In [ ]:
# =====================================================================
# REPORT JSON ASSEMBLY
# Combines: normalization + data quality + metrics + chart data +
# insights (deterministic or LLM) into one dict matching ReportDocument.
# =====================================================================

REPORT_TITLES = {
    "machine_performance": "Machine Performance Report",
    "maintenance": "Maintenance Report",
    "production": "Production Report",
    "safety_video_analytics": "Safety / Video Analytics Report",
    "cross_source_asset": "Cross-Source Asset Report",
    "executive_summary": "Executive Summary Report",
}


def _time_range_from_frames(normalized: Dict[str, Any]) -> Dict[str, Optional[str]]:
    mins, maxs = [], []
    for key, df in normalized["source_frames"].items():
        ts_col = normalized["timestamp_cols"].get(key)
        if ts_col and ts_col in df.columns and not df.empty:
            valid = df[ts_col].dropna()
            if not valid.empty:
                mins.append(valid.min())
                maxs.append(valid.max())
    if not mins:
        return {"start": None, "end": None}
    return {"start": min(mins).isoformat(), "end": max(maxs).isoformat()}


def generate_report(normalized_result: Dict[str, Any], report_type: Optional[str] = None) -> Dict[str, Any]:
    """
    End-to-end: normalized_result (NormalizedResearchResult dict) -> full
    report dict matching the ReportDocument contract. Pure/deterministic
    aside from the optional insight step (which itself always falls back
    to deterministic on failure).
    """
    research_id = normalized_result.get("research_id", f"research_{uuid.uuid4().hex[:8]}")
    normalized = normalize_research_result(normalized_result)
    data_quality = compute_data_quality(normalized)

    kpis: List[Dict[str, Any]] = []
    for key, df in normalized["source_frames"].items():
        kpis.extend(build_kpis_for_source(key, df, normalized["timestamp_cols"].get(key)))

    charts = build_all_charts(normalized)

    tables = [
        {
            "id": f"table_{key.replace('.', '_')}",
            "title": f"{key.split('.')[-1].replace('_', ' ').title()} - Sample Records",
            "columns": list(df.columns),
            "rows": json.loads(
                df.head(20).astype(object).where(pd.notna(df.head(20)), None)
                .to_json(orient="records", date_format="iso")
            ) if not df.empty else [],
        }
        for key, df in normalized["source_frames"].items()
    ]

    report_type = report_type or detect_report_type(normalized_result)

    context = {
        "report_type": report_type,
        "sources": list(normalized["source_frames"].keys()),
        "data_quality": data_quality,
        "kpis": kpis,
    }
    insights, llm_used = generate_insights(context)

    calculation_notes = [
        "All numeric KPIs computed with pandas (mean/median/min/max/std/trend) -- no LLM involved.",
        "Chart data uses structured objects per point, never stringified coordinate lists.",
    ]
    if not any(s.get("status") == "success" for s in normalized_result.get("sources", [])):
        calculation_notes.append(
            "No source reported status='success': this report may be based on query "
            "plans that have not yet been executed against a live database, or on demo/synthetic data."
        )

    time_range = _time_range_from_frames(normalized)

    report_dict = {
        "report": {
            "report_id": f"rpt_{uuid.uuid4().hex[:10]}",
            "title": REPORT_TITLES.get(report_type, "MAI Report"),
            "report_type": report_type,
            "status": "success" if data_quality["source_records"] > 0 else "partial",
            "generated_at": datetime.now(timezone.utc).isoformat(),
            "time_range": {**time_range, "timezone": DEFAULT_TIMEZONE},
        },
        "summary": insights,
        "kpis": kpis,
        "charts": charts,
        "tables": tables,
        "data_quality": data_quality,
        "sources": normalized_result.get("sources", []),
        "metadata": {
            "agent": REPORT_AGENT_NAME,
            "research_id": research_id,
            "report_version": REPORT_VERSION,
            "calculation_notes": calculation_notes,
            "llm_used": llm_used,
        },
    }
    return report_dict


# Smoke-check
_report_dict = generate_report(_demo)
print("report_type:", _report_dict["report"]["report_type"])
print("kpis:", len(_report_dict["kpis"]), "charts:", len(_report_dict["charts"]), "tables:", len(_report_dict["tables"]))


## Report Validation

In [ ]:
# =====================================================================
# REPORT VALIDATION
# Validates the assembled dict against the ReportDocument Pydantic
# contract. Returns structured errors instead of crashing.
# =====================================================================

def validate_report(report_dict: Dict[str, Any]) -> Tuple[bool, Optional[ReportDocument], List[str]]:
    try:
        doc = ReportDocument.model_validate(report_dict)
        return True, doc, []
    except ValidationError as e:
        errors = [str(err) for err in e.errors()]
        return False, None, errors
    except Exception as e:  # defensive: never let validation itself crash the pipeline
        return False, None, [f"Unexpected validation error: {e}"]


# Smoke-check
_is_valid, _report_model, _errors = validate_report(_report_dict)
print("valid:", _is_valid, "errors:", _errors[:5])


## Report JSON Pretty-Print

In [ ]:
# =====================================================================
# REPORT JSON PRETTY-PRINT + SAVE
# =====================================================================

def save_report_json(report_dict: Dict[str, Any], path: Path = REPORT_JSON_PATH) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(report_dict, f, indent=2, default=str)
    return path


# Smoke-check: pretty-print a trimmed preview + save full file
_preview = {
    "report": _report_dict["report"],
    "summary": {"executive_summary": _report_dict["summary"]["executive_summary"],
                "key_findings": _report_dict["summary"]["key_findings"][:2],
                "recommendations": _report_dict["summary"]["recommendations"][:2]},
    "kpis": _report_dict["kpis"][:2],
    "charts": [{"id": c["id"], "chart_type": c["chart_type"], "title": c["title"]} for c in _report_dict["charts"]],
    "data_quality": _report_dict["data_quality"],
}
print(json.dumps(_preview, indent=2, default=str))
_saved_path = save_report_json(_report_dict)
print(f"\nSaved full report JSON -> {_saved_path.resolve()}")


## Chart Rendering

In [ ]:
# =====================================================================
# CHART RENDERING (matplotlib -> PNG)
# Kept separate from PDF assembly on purpose (cell 19 just places these
# image files). Never crashes the pipeline: failures are captured and
# returned so the PDF step can render an "unavailable" placeholder.
# =====================================================================

CHART_COLORS = ["#2563eb", "#dc2626", "#16a34a", "#d97706", "#7c3aed", "#0891b2"]


def render_chart(chart: Dict[str, Any], out_dir: Path = CHARTS_DIR) -> Optional[Path]:
    """Render one Chart-contract dict to a PNG. Returns None for 'table'/'kpi'
    chart types (rendered as text/tables directly in the PDF, not images)."""
    chart_type = chart["chart_type"]
    if chart_type in ("table", "kpi") or not chart["data"]:
        return None

    out_path = out_dir / f"{chart['id']}.png"
    fig, ax = plt.subplots(figsize=(6.5, 3.6), dpi=150)

    try:
        if chart_type == "line":
            for i, series in enumerate(chart["series"]):
                xs = [pd.to_datetime(d["timestamp"]) for d in chart["data"]]
                ys = [d.get(series["field"]) for d in chart["data"]]
                ax.plot(xs, ys, label=series["name"], color=CHART_COLORS[i % len(CHART_COLORS)], linewidth=1.5)
            ax.set_xlabel(chart["x_axis"]["label"])
            ax.set_ylabel(chart["y_axis"]["label"])
            ax.legend(fontsize=8, loc="upper left")
            fig.autofmt_xdate()

        elif chart_type == "bar":
            cats = [d["category"] for d in chart["data"]]
            vals = [d["value"] for d in chart["data"]]
            ax.bar(cats, vals, color=CHART_COLORS[0])
            ax.set_xlabel(chart["x_axis"]["label"])
            ax.set_ylabel(chart["y_axis"]["label"])
            plt.setp(ax.get_xticklabels(), rotation=30, ha="right", fontsize=8)

        elif chart_type == "pie":
            labels = [d["category"] for d in chart["data"]]
            vals = [d["value"] for d in chart["data"]]
            ax.pie(vals, labels=labels, autopct="%1.0f%%", colors=CHART_COLORS,
                   textprops={"fontsize": 8})
            ax.axis("equal")

        else:
            plt.close(fig)
            return None

        ax.set_title(chart["title"], fontsize=10)
        fig.tight_layout()
        fig.savefig(out_path)
        return out_path
    except Exception as e:
        warnings.warn(f"Failed to render chart '{chart.get('id')}': {e}")
        return None
    finally:
        plt.close(fig)


def render_all_charts(report_dict: Dict[str, Any]) -> Dict[str, Optional[str]]:
    """Render every chart in the report; returns {chart_id: png_path_or_None}."""
    paths: Dict[str, Optional[str]] = {}
    for chart in report_dict.get("charts", []):
        p = render_chart(chart)
        paths[chart["id"]] = str(p) if p else None
    return paths


# Smoke-check
_chart_paths = render_all_charts(_report_dict)
for cid, p in _chart_paths.items():
    print(cid, "->", p)


## PDF Generation

In [ ]:
# =====================================================================
# PDF GENERATION (ReportLab)
# Cover -> metadata -> executive summary -> KPI cards -> key findings ->
# charts -> tables -> recommendations -> data quality -> sources ->
# calculation notes. Every section degrades gracefully if empty.
# =====================================================================

_styles = getSampleStyleSheet()
_styles.add(ParagraphStyle(name="MAITitle", fontSize=22, leading=26, spaceAfter=6, alignment=TA_CENTER))
_styles.add(ParagraphStyle(name="MAISubtitle", fontSize=12, leading=16, textColor=colors.grey, alignment=TA_CENTER))
_styles.add(ParagraphStyle(name="MAISection", fontSize=14, leading=18, spaceBefore=14, spaceAfter=8,
                            textColor=colors.HexColor("#1e3a8a")))
_styles.add(ParagraphStyle(name="MAIBody", fontSize=9.5, leading=13))
_styles.add(ParagraphStyle(name="MAISmall", fontSize=8, leading=11, textColor=colors.grey))

_STATUS_COLORS = {
    "normal": colors.HexColor("#16a34a"), "warning": colors.HexColor("#d97706"),
    "critical": colors.HexColor("#dc2626"), "unknown": colors.grey,
    "info": colors.HexColor("#2563eb"),
}


def _p(text: str, style: str = "MAIBody") -> Paragraph:
    return Paragraph(str(text).replace("\n", "<br/>"), _styles[style])


def _section_header(title: str):
    return [Paragraph(title, _styles["MAISection"]), HRFlowable(width="100%", color=colors.HexColor("#cbd5e1"))]


def _kpi_table(kpis: List[Dict[str, Any]]):
    if not kpis:
        return [_p("No KPIs available for this report.", "MAISmall")]
    rows = [["KPI", "Value", "Trend", "Status"]]
    for k in kpis:
        comp = k.get("comparison") or {}
        trend = f"{comp.get('value')}% ({comp.get('direction')})" if comp.get("value") is not None else "n/a"
        val = f"{k['value']}" if k.get("value") is not None else "n/a"
        if k.get("unit"):
            val += f" {k['unit']}"
        rows.append([_p(k["label"], "MAIBody"), val, trend, k.get("status", "unknown")])
    t = RLTable(rows, colWidths=[7 * cm, 3 * cm, 3.5 * cm, 2.5 * cm], repeatRows=1)
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1e3a8a")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTSIZE", (0, 0), (-1, -1), 8.5),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#e2e8f0")),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f8fafc")]),
    ]))
    return [t]


def _findings_flow(findings: List[Dict[str, Any]]):
    if not findings:
        return [_p("No key findings.", "MAISmall")]
    flows = []
    for f in findings:
        color = _STATUS_COLORS.get(f.get("severity", "info"), colors.grey)
        flows.append(Paragraph(f"<font color='{color.hexval()}'>&#9679;</font> <b>{f['title']}</b>", _styles["MAIBody"]))
        flows.append(_p(f["description"], "MAISmall"))
        flows.append(Spacer(1, 4))
    return flows


def _recommendations_flow(recs: List[Dict[str, Any]]):
    if not recs:
        return [_p("No recommendations.", "MAISmall")]
    flows = []
    for r in recs:
        flows.append(_p(f"<b>[{r.get('priority', 'medium').upper()}] {r['title']}</b> ({r.get('action_type', '')})"))
        flows.append(_p(r["description"], "MAISmall"))
        flows.append(Spacer(1, 4))
    return flows


def _data_table_flow(table: Dict[str, Any], max_cols: int = 6, max_rows: int = 10):
    cols = table["columns"][:max_cols]
    rows = table["rows"][:max_rows]
    if not rows:
        return [_p("No rows available.", "MAISmall")]
    header = [_p(f"<b>{c}</b>", "MAISmall") for c in cols]
    data = [header]
    for r in rows:
        data.append([_p(str(r.get(c, ""))[:40], "MAISmall") for c in cols])
    col_width = (17 * cm) / max(1, len(cols))
    t = RLTable(data, colWidths=[col_width] * len(cols), repeatRows=1)
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#334155")),
        ("GRID", (0, 0), (-1, -1), 0.3, colors.HexColor("#e2e8f0")),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f8fafc")]),
    ]))
    note = []
    if len(table["rows"]) > max_rows or len(table["columns"]) > max_cols:
        note = [_p(f"Showing {min(max_rows, len(table['rows']))} of {len(table['rows'])} row(s), "
                    f"{min(max_cols, len(table['columns']))} of {len(table['columns'])} column(s). "
                    f"Full data is in report.json.", "MAISmall")]
    return [t] + note


def build_pdf_report(report_dict: Dict[str, Any], chart_paths: Dict[str, Optional[str]],
                      out_path: Path = PDF_PATH) -> Path:
    """Assemble the full PDF from the validated report dict. Never raises
    on missing/empty sections -- degrades gracefully."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    doc = SimpleDocTemplate(str(out_path), pagesize=A4,
                             leftMargin=1.5 * cm, rightMargin=1.5 * cm,
                             topMargin=1.5 * cm, bottomMargin=1.5 * cm)
    story = []
    meta = report_dict["report"]

    # 1. Cover / title
    story.append(Spacer(1, 3 * cm))
    story.append(Paragraph("MAI - Manufacturing Agentic AI", _styles["MAISubtitle"]))
    story.append(Paragraph(meta["title"], _styles["MAITitle"]))
    story.append(Paragraph(f"Report type: {meta['report_type'].replace('_', ' ').title()}", _styles["MAISubtitle"]))
    story.append(Spacer(1, 1 * cm))
    story.append(_p(f"Generated: {meta['generated_at']}", "MAISmall"))
    tr = meta.get("time_range", {})
    if tr.get("start") and tr.get("end"):
        story.append(_p(f"Data window: {tr['start']} &ndash; {tr['end']} ({tr.get('timezone', '')})", "MAISmall"))
    story.append(_p(f"Report ID: {meta['report_id']} | Status: {meta['status']}", "MAISmall"))
    story.append(PageBreak())

    # 2. Report metadata (kept brief; full detail lives in the metadata section later)
    story += _section_header("Report Metadata")
    story.append(_p(f"Research ID: {report_dict['metadata']['research_id']} | "
                     f"Report version: {report_dict['metadata']['report_version']} | "
                     f"LLM narrative used: {report_dict['metadata']['llm_used']}"))

    # 3. Executive summary
    story += _section_header("Executive Summary")
    story.append(_p(report_dict["summary"]["executive_summary"]))

    # 4. KPI cards
    story += _section_header("Key Performance Indicators")
    story += _kpi_table(report_dict["kpis"])

    # 5. Key findings
    story += _section_header("Key Findings")
    story += _findings_flow(report_dict["summary"]["key_findings"])

    # 6. Charts
    story += _section_header("Charts")
    any_chart = False
    for chart in report_dict["charts"]:
        img_path = chart_paths.get(chart["id"])
        if img_path and Path(img_path).exists():
            any_chart = True
            story.append(KeepTogether([
                _p(f"<b>{chart['title']}</b>"),
                RLImage(img_path, width=15 * cm, height=8.3 * cm),
                Spacer(1, 6),
            ]))
    if not any_chart:
        story.append(_p("No chart images available for this report.", "MAISmall"))

    # 7. Data tables
    story += _section_header("Data Tables")
    if report_dict["tables"]:
        for table in report_dict["tables"]:
            story.append(_p(f"<b>{table['title']}</b>"))
            story += _data_table_flow(table)
            story.append(Spacer(1, 8))
    else:
        story.append(_p("No tables available.", "MAISmall"))

    # 8. Recommendations
    story += _section_header("Recommendations")
    story += _recommendations_flow(report_dict["summary"]["recommendations"])

    # 9. Data quality
    story += _section_header("Data Quality")
    dq = report_dict["data_quality"]
    story.append(_p(f"Status: <b>{dq['status']}</b> | Source records: {dq['source_records']} | "
                     f"Missing values: {dq['missing_values']} | Invalid values: {dq['invalid_values']}"))
    if dq["warnings"]:
        for w in dq["warnings"]:
            story.append(_p(f"&#8226; {w}", "MAISmall"))
    else:
        story.append(_p("No data-quality warnings.", "MAISmall"))

    # 10. Source / provenance
    story += _section_header("Sources / Provenance")
    if report_dict["sources"]:
        rows = [["Database", "Source", "Records", "Status"]]
        for s in report_dict["sources"]:
            rows.append([s.get("database", ""), s.get("source_name", ""),
                         str(s.get("record_count", 0)), s.get("status", "")])
        t = RLTable(rows, colWidths=[4 * cm, 6 * cm, 3 * cm, 5 * cm], repeatRows=1)
        t.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#334155")),
            ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
            ("FONTSIZE", (0, 0), (-1, -1), 8.5),
            ("GRID", (0, 0), (-1, -1), 0.3, colors.HexColor("#e2e8f0")),
        ]))
        story.append(t)
    else:
        story.append(_p("No source provenance recorded.", "MAISmall"))

    # 11. Calculation notes
    notes = report_dict["metadata"].get("calculation_notes", [])
    if notes:
        story += _section_header("Calculation Notes")
        for n in notes:
            story.append(_p(f"&#8226; {n}", "MAISmall"))

    try:
        doc.build(story)
    except Exception as e:
        raise RuntimeError(f"PDF generation failed: {e}") from e

    return out_path


# Smoke-check
_pdf_path = build_pdf_report(_report_dict, _chart_paths)
print("PDF written:", _pdf_path.resolve(), "size:", _pdf_path.stat().st_size, "bytes")


## PDF Validation / Checks

In [ ]:
# =====================================================================
# PDF VALIDATION / CHECKS
# Lightweight, dependency-optional sanity checks on the generated PDF.
# =====================================================================

try:
    from pypdf import PdfReader
    PYPDF_AVAILABLE = True
except ImportError:
    PYPDF_AVAILABLE = False


def validate_pdf(path: Path) -> Dict[str, Any]:
    result = {"path": str(path), "exists": False, "non_empty": False,
              "valid_pdf_header": False, "page_count": None, "issues": []}

    if not path.exists():
        result["issues"].append("File does not exist.")
        return result
    result["exists"] = True

    size = path.stat().st_size
    result["non_empty"] = size > 0
    if not result["non_empty"]:
        result["issues"].append("File is empty (0 bytes).")

    with open(path, "rb") as f:
        header = f.read(5)
    result["valid_pdf_header"] = header == b"%PDF-"
    if not result["valid_pdf_header"]:
        result["issues"].append("File does not start with a valid %PDF- header.")

    if PYPDF_AVAILABLE and result["valid_pdf_header"]:
        try:
            reader = PdfReader(str(path))
            result["page_count"] = len(reader.pages)
            if result["page_count"] == 0:
                result["issues"].append("PDF has 0 pages.")
        except Exception as e:
            result["issues"].append(f"pypdf failed to parse PDF: {e}")
    elif not PYPDF_AVAILABLE:
        result["issues"].append("pypdf not installed -- skipped page-count check (non-fatal).")

    result["ok"] = result["exists"] and result["non_empty"] and result["valid_pdf_header"] and \
        (result["page_count"] is None or result["page_count"] > 0)
    return result


# Smoke-check
_pdf_check = validate_pdf(_pdf_path)
print(json.dumps(_pdf_check, indent=2))


## End-to-End Demo

In [ ]:
# =====================================================================
# END-TO-END DEMO
# Uses live environment-backed data only. No synthetic insertions are created.
# =====================================================================

def run_pipeline(normalized_result: Dict[str, Any], report_type: Optional[str] = None,
                  save_outputs: bool = True) -> Dict[str, Any]:
    """One call that runs stages 2-13 of the architecture end-to-end."""
    report_dict = generate_report(normalized_result, report_type=report_type)
    is_valid, report_model, errors = validate_report(report_dict)

    result = {"report_dict": report_dict, "is_valid": is_valid, "validation_errors": errors,
               "json_path": None, "chart_paths": {}, "pdf_path": None, "pdf_check": None}

    if not is_valid:
        warnings.warn(f"Report failed validation: {errors}")
        return result

    if save_outputs:
        result["json_path"] = str(save_report_json(report_dict))
        result["chart_paths"] = render_all_charts(report_dict)
        pdf_path = build_pdf_report(report_dict, result["chart_paths"])
        result["pdf_path"] = str(pdf_path)
        result["pdf_check"] = validate_pdf(pdf_path)

    return result


print("=" * 70)
print("DEMO: Machine Performance Report (LIVE DATA FROM ENV)")
print("=" * 70)

demo_normalized = build_live_normalized_result("machine_performance")
demo_result = run_pipeline(demo_normalized)

print(f"Report type      : {demo_result['report_dict']['report']['report_type']}")
print(f"Valid             : {demo_result['is_valid']}")
print(f"KPIs generated    : {len(demo_result['report_dict']['kpis'])}")
print(f"Charts generated  : {len(demo_result['report_dict']['charts'])}")
print(f"Data quality      : {demo_result['report_dict']['data_quality']['status']}")
print(f"Report JSON       : {demo_result['json_path']}")
print(f"PDF report        : {demo_result['pdf_path']}")
print(f"PDF check         : {demo_result['pdf_check']}")
print()
print("Executive summary:")
print(" ", demo_result["report_dict"]["summary"]["executive_summary"])


## Test Cases

In [ ]:
# =====================================================================
# TEST CASES
# Assertion-based smoke tests covering every report type plus edge
# cases (empty data, missing timestamps, malformed rows). Prints a
# PASS/FAIL summary; does not raise on the first failure so the full
# suite always runs.
# =====================================================================

def _run_test(name: str, fn) -> bool:
    try:
        fn()
        print(f"  PASS  {name}")
        return True
    except AssertionError as e:
        print(f"  FAIL  {name}: {e}")
        return False
    except Exception as e:
        print(f"  ERROR {name}: {type(e).__name__}: {e}")
        return False


def test_all_report_types():
    for report_type in REPORT_TITLES:
        norm = build_synthetic_normalized_result(report_type)
        result = run_pipeline(norm, report_type=report_type, save_outputs=False)
        assert result["is_valid"], f"{report_type}: validation errors {result['validation_errors']}"
        assert result["report_dict"]["report"]["report_type"] == report_type


def test_empty_research_result():
    empty = {"research_id": "test_empty", "user_query": "test", "sources": [],
             "data": [], "columns": [], "metadata": {}, "data_quality": {}}
    result = run_pipeline(empty, report_type="executive_summary", save_outputs=False)
    assert result["is_valid"], f"Empty result should still validate: {result['validation_errors']}"
    assert result["report_dict"]["data_quality"]["source_records"] == 0
    assert result["report_dict"]["data_quality"]["status"] in ("poor", "unknown")


def test_missing_timestamps():
    rows = [{"machine_id": "MC-1", "temperature": 70, "__source_database": "influxdb",
              "__source_table": "machine_telemetry"} for _ in range(5)]  # no timestamp field at all
    norm = {"research_id": "test_no_ts", "user_query": "machine performance", "sources": [
        {"database": "influxdb", "source_name": "machine_telemetry", "record_count": 5, "status": "success"}],
        "data": rows, "columns": ["machine_id", "temperature"], "metadata": {}, "data_quality": {}}
    result = run_pipeline(norm, save_outputs=False)
    assert result["is_valid"]
    # No timestamp column -> no line chart should be produced, but bar/table should still exist
    chart_types = {c["chart_type"] for c in result["report_dict"]["charts"]}
    assert "line" not in chart_types, "Should not produce a line chart with no timestamp column"


def test_malformed_rows_do_not_crash():
    rows = [
        {"machine_id": "MC-1", "temperature": None, "timestamp": "not-a-date",
         "__source_database": "influxdb", "__source_table": "machine_telemetry"},
        {"machine_id": "MC-2", "temperature": -999, "timestamp": "2026-09-24T09:00:00Z",
         "__source_database": "influxdb", "__source_table": "machine_telemetry"},
    ]
    norm = {"research_id": "test_malformed", "user_query": "machine performance", "sources": [
        {"database": "influxdb", "source_name": "machine_telemetry", "record_count": 2, "status": "success"}],
        "data": rows, "columns": ["machine_id", "temperature", "timestamp"], "metadata": {}, "data_quality": {}}
    result = run_pipeline(norm, save_outputs=False)
    assert result["is_valid"]
    dq = result["report_dict"]["data_quality"]
    assert dq["source_records"] == 2
    assert any("negative" in w.lower() for w in dq["warnings"]), "Negative value should be flagged, not silently kept"


def test_chart_type_validation_rejects_bad_pie():
    df = pd.DataFrame({"category": [f"c{i}" for i in range(20)], "value": range(20)})
    bad_spec = {"id": "x", "chart_type": "pie", "x_field": None, "y_fields": [], "cat_field": "category"}
    assert validate_chart_spec(bad_spec, df) is False, "20-category pie chart should be rejected"


def test_pdf_generation_with_zero_charts():
    norm = {"research_id": "test_zero_charts", "user_query": "test", "sources": [], "data": [],
            "columns": [], "metadata": {}, "data_quality": {}}
    report_dict = generate_report(norm, report_type="executive_summary")
    pdf_path = build_pdf_report(report_dict, {}, out_path=OUTPUT_DIR / "test_zero_charts.pdf")
    check = validate_pdf(pdf_path)
    assert check["ok"], f"PDF with zero charts should still build successfully: {check}"


def test_report_json_matches_pydantic_contract():
    norm = build_synthetic_normalized_result("safety_video_analytics")
    report_dict = generate_report(norm)
    is_valid, model, errors = validate_report(report_dict)
    assert is_valid, errors
    assert model.report.report_type == "safety_video_analytics"


TESTS = [
    ("All report types produce a valid report", test_all_report_types),
    ("Empty research result handled gracefully", test_empty_research_result),
    ("Missing timestamps do not produce line charts", test_missing_timestamps),
    ("Malformed/anomalous rows do not crash the pipeline", test_malformed_rows_do_not_crash),
    ("Chart validation rejects high-cardinality pie charts", test_chart_type_validation_rejects_bad_pie),
    ("PDF builds successfully with zero charts", test_pdf_generation_with_zero_charts),
    ("Report JSON matches the Pydantic contract", test_report_json_matches_pydantic_contract),
]

print("=" * 70)
print("TEST SUITE")
print("=" * 70)
_results = [_run_test(name, fn) for name, fn in TESTS]
print("-" * 70)
print(f"{sum(_results)}/{len(_results)} tests passed")


## Integration instructions for the existing MAI backend

1. **Point `RESEARCHER_MODULE_NAME` (Cell 6) at your real Researcher Agent file** (e.g. `researcher_agent.py`)
   and make it importable (same directory / `PYTHONPATH`). No changes to that file are required.
2. **Call the existing pipeline, unchanged**, then adapt its output:
   ```python
   researcher_out = run_researcher_pipeline("Show me machine utilization for last week")
   normalized = adapt_researcher_output(researcher_out["sql_result"])
   report = run_pipeline(normalized)  # report["json_path"], report["pdf_path"]
   ```
3. **Once a database-execution layer exists**, pass its results into the adapter so sources move from
   `"query_generated_not_executed"` to `"success"` with real rows — no other code changes needed:
   ```python
   execution_results = {"mes.MachineMaster": [...rows...]}
   normalized = adapt_researcher_output(researcher_out["sql_result"], execution_results=execution_results)
   ```
4. **To enable the optional LLM narrative layer**, set `USE_LLM = True` (Cell 3) and configure
   `MAI_LLM_MODEL` / `MAI_LLM_API_BASE` / `MAI_LLM_API_KEY` as environment variables — no code changes.
5. **Frontend integration**: `report["report_dict"]` (or `outputs/report_generator/report.json`) is the
   full `ReportDocument` contract (Cell 4) — hand it directly to a frontend report renderer.

---

## NEXT IMPLEMENTATION STEPS FOR MAI

- **Database execution layer**: the Researcher Agent currently only plans/generates SQL. A layer that
  actually executes the generated SQL against SQL Server (MES) and PostgreSQL (Video Analytics) and
  returns rows is required before `adapt_researcher_output(execution_results=...)` sees real data.
- **InfluxDB integration**: a separate Flux-query generation/execution mechanism is needed for
  time-series telemetry (explicitly NOT forced into the SQL generator, per project requirements). This
  notebook's `NormalizedResearchResult` contract already treats `"influxdb"` as a first-class
  `database` value, so the normalization/quality/metrics/chart layers need no changes once that
  mechanism exists.
- **Relationship-aware query generation**: the provided schema files carry no foreign-key/relationship
  metadata, so the current Researcher Agent selects tables independently rather than generating joins.
  Cross-source/cross-table reports in this notebook combine independently-queried result sets; a
  join-planning layer (and richer schema metadata) is needed for true relational joins.
- **Stronger SQL/Flux validation**: `validate_sql()` in the Researcher Agent is a lightweight heuristic
  check (ORDER BY presence, column count, `SELECT *`, residual noise columns). It does not verify
  dialect-specific syntax (e.g. SQL Server `TOP`/`OFFSET-FETCH` vs. PostgreSQL `LIMIT`, quoting rules,
  reserved words). A dialect-aware validator (or a real `EXPLAIN`/dry-run against each DB) is recommended
  before these queries reach production.
- **Real Researcher Agent adapter hardening**: `adapt_researcher_output` currently keys execution
  results by `"{database}.{table}"` or `"{table}"`. Once the execution layer's actual response shape is
  known, confirm this key format and extend the adapter for partial-failure responses (e.g. one table's
  query fails while others succeed).
- **LangGraph integration**: this notebook's stages (normalize → quality → metrics → chart-select →
  insights → assemble → validate → render → PDF) are already function-per-stage and map cleanly onto
  LangGraph nodes; wiring them into the existing MAI LangGraph orchestration is the next step.
- **LiteLLM optional narrative layer**: the structured-output prompt/parsing in Cell 14 is implemented
  and has a tested deterministic fallback, but has not been exercised against a live LiteLLM endpoint
  in this environment (no network access here). Test against your actual provider before enabling
  `USE_LLM = True` in production.
- **Production PDF storage/download integration**: `build_pdf_report()` writes to a local path
  (`outputs/report_generator/MAI_Report.pdf`). Wire this into whatever object storage / download
  mechanism the MAI backend uses (e.g. upload the returned `Path` to S3/Blob storage and return a
  signed URL to the frontend).
- **Frontend report renderer integration**: `ReportDocument` (Cell 4) is the frontend contract. Confirm
  it against the actual frontend renderer's expected shape and version it (`report_version`) if it needs
  to change.
- **Migrating to production Python modules**: each cell in this notebook is already a self-contained,
  type-hinted, docstringed module (config / models / adapter / normalize / quality / metrics /
  report_type / chart_selection / chart_data / llm / assembly / validation / rendering / pdf). Splitting
  them into files of the same name is a mechanical refactor.

---

### Summary

- **Reused from the Researcher Agent**: schema loading/cleaning, domain-vocabulary table/column
  selection, date detection, SQL generation, SQL validation — wrapped, not rewritten, via
  `run_researcher_pipeline()` / `adapt_researcher_output()`.
- **Implemented now**: the full Report Generator + PDF pipeline (Pydantic contracts, data normalization,
  data-quality engine, deterministic metrics engine, report-type detection, deterministic chart
  selection + structured chart data, optional provider-agnostic LLM narrative layer with deterministic
  fallback, report JSON assembly + validation, matplotlib chart rendering, a professional multi-section
  ReportLab PDF, PDF validation, an end-to-end demo, and a 7-case test suite — all verified executable
  end-to-end on synthetic data in this environment).
- **Remaining production integration**: see the list above — primarily the DB execution layer, InfluxDB/
  Flux support, join-aware querying, and wiring into LangGraph / live LiteLLM / storage / frontend.
